# S9 — AndinaLog 03B | MLP inicial

Se entrena una red 32→16 para anticipar desviaciones térmicas. La arquitectura es una hipótesis inicial sin regularización ni parada temprana.

## 1. Entorno y datos

Se reutiliza el split temporal congelado. La semilla mejora la reproducibilidad, aunque versiones y hardware pueden producir pequeñas diferencias.

In [3]:
from pathlib import Path
import os
import sys
import random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_DETERMINISTIC_OPS"] = "1"
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

ENTORNO = "auto"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
SEMILLA = 42
TARGET = "clasificacion_objetivo_60min"
random.seed(SEMILLA); np.random.seed(SEMILLA); tf.random.set_seed(SEMILLA)

def encontrar_raiz():
    if ENTORNO == "drive" or (ENTORNO == "auto" and "google.colab" in sys.modules):
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(RUTA_PROYECTO_DRIVE)
    else:
        raiz = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "proyecto-integrador/03_EDA/salidas_v4_eventos/andinalog_03b_evidencia_3_lecturas_con_eventos.csv").is_file()), None)
    if raiz is None: raise FileNotFoundError("No se encontró la raíz del proyecto")
    return raiz

RAIZ = encontrar_raiz()
RUTA_DATOS = RAIZ / "proyecto-integrador/03_EDA/salidas_v4_eventos/andinalog_03b_evidencia_3_lecturas_con_eventos.csv"
RUTA_SPLIT = RAIZ / "proyecto-integrador/04_regresion/salidas_s6/asignacion_split_viajes.csv"
RUTA_S7 = RAIZ / "proyecto-integrador/05_clasificacion/salidas_s7/metricas_validacion_clasificacion_s7.csv"
SALIDAS = RAIZ / "proyecto-integrador/07_mlp/salidas_s9"
df = pd.read_csv(RUTA_DATOS, encoding="utf-8-sig")
split = pd.read_csv(RUTA_SPLIT, encoding="utf-8-sig")
df["particion"] = df["viaje_id"].map(split.set_index("viaje_id")["particion"])
df["apta_clasificacion_60min"] = df["apta_clasificacion_60min"].astype("boolean")
modelado = df.loc[df["apta_clasificacion_60min"].fillna(False) & df[TARGET].isin([0,1])].copy()
modelado[TARGET] = modelado[TARGET].astype(int)
print("TensorFlow:", tf.__version__, "| lecturas aptas:", len(modelado))


TensorFlow: 2.20.0 | lecturas aptas: 28557


## 2. Contrato y particiones

La clase positiva es una desviación en los próximos 60 minutos. TEST se prepara, pero permanece sellado durante S9.

In [4]:
NUMERICAS = [
    "temp_c", "humedad_pct", "objetivo_c", "tolerancia_c", "desvio_respecto_umbral_c",
    "temp_lag1_c", "temp_lag2_c", "cambio_temp_c", "pendiente_c_por_min",
    "temp_media_historica_3", "temp_max_historica_3", "minutos_desde_inicio",
    "capacidad_kg_tratada", "cantidad_solicitada_tratado", "tiempo_entrega_prometido_hrs_tratado",
]
CATEGORICAS = ["categoria_logistica_tratada", "tipo_camion_tratado", "centro_distribucion_tratado"]
FEATURES = NUMERICAS + CATEGORICAS
PROHIBIDAS = {TARGET, "max_desvio_termico_proximos_60min_c", "n_lecturas_futuras_60min",
              "apta_clasificacion_60min", "apta_regresion_60min"}
assert not set(FEATURES) & PROHIBIDAS

ajuste = modelado.loc[modelado["particion"].eq("AJUSTE")].copy()
validacion = modelado.loc[modelado["particion"].eq("VALIDACION")].copy()
test_sellado = modelado.loc[modelado["particion"].eq("TEST")].copy()
assert set(ajuste["viaje_id"]).isdisjoint(validacion["viaje_id"])
assert set(ajuste["viaje_id"]).isdisjoint(test_sellado["viaje_id"])
print("Ajuste/validación/test sellado:", len(ajuste), len(validacion), len(test_sellado))
print("Prevalencias:", ajuste[TARGET].mean(), validacion[TARGET].mean(), test_sellado[TARGET].mean())


Ajuste/validación/test sellado: 18946 4287 4274
Prevalencias: 0.046236672648580175 0.03522276650338232 0.0542817033224146


## 3. Preprocesamiento seguro y baseline

Imputación, escalamiento y OneHotEncoder se ajustan únicamente con AJUSTE y luego se aplican a validación y test.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             average_precision_score, roc_auc_score, confusion_matrix)

preprocesador = ColumnTransformer([
    ("num", Pipeline([("imputar", SimpleImputer(strategy="median", add_indicator=True)),
                      ("escalar", StandardScaler())]), NUMERICAS),
    ("cat", Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), CATEGORICAS),
], sparse_threshold=0)
X_aj = preprocesador.fit_transform(ajuste[FEATURES]).astype("float32")
X_val = preprocesador.transform(validacion[FEATURES]).astype("float32")
X_test_preparado = preprocesador.transform(test_sellado[FEATURES]).astype("float32")
y_aj = ajuste[TARGET].to_numpy(dtype="float32")
y_val = validacion[TARGET].to_numpy(dtype="float32")
y_test_sellado = test_sellado[TARGET].to_numpy(dtype="float32")
assert np.isfinite(X_aj).all() and np.isfinite(X_val).all() and np.isfinite(X_test_preparado).all()
print("Dimensión después del preprocesamiento:", X_aj.shape[1])

def metricas(real, prob, umbral=.5):
    pred = (prob >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(real, pred, labels=[0,1]).ravel()
    return {"accuracy": accuracy_score(real, pred), "precision": precision_score(real, pred, zero_division=0),
            "recall": recall_score(real, pred, zero_division=0), "f1": f1_score(real, pred, zero_division=0),
            "pr_auc": average_precision_score(real, prob), "roc_auc": roc_auc_score(real, prob),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp}

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_aj, y_aj)
prob_dummy = dummy.predict_proba(X_val)[:,1]
metricas_dummy = metricas(y_val, prob_dummy)
print("Baseline validación:", metricas_dummy)


Dimensión después del preprocesamiento: 31
Baseline validación: {'accuracy': 0.9647772334966177, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'pr_auc': np.float64(0.03522276650338232), 'roc_auc': np.float64(0.5), 'TN': np.int64(4136), 'FP': np.int64(0), 'FN': np.int64(151), 'TP': np.int64(0)}


## 4. Arquitectura inicial

La MLP usa capas de 32 y 16 neuronas ReLU y salida sigmoide. Se entrena durante 20 épocas sin EarlyStopping para observar las curvas.

In [6]:
# Arquitectura inicial de S9: sin Dropout, L2 ni EarlyStopping.
mlp = keras.Sequential([
    keras.Input(shape=(X_aj.shape[1],), name="entrada"),
    layers.Dense(32, activation="relu", name="oculta_1"),
    layers.Dense(16, activation="relu", name="oculta_2"),
    layers.Dense(1, activation="sigmoid", name="probabilidad_desviacion"),
], name="MLP_AndinaLog_S9")
mlp.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="binary_crossentropy",
            metrics=[keras.metrics.BinaryAccuracy(name="accuracy"), keras.metrics.Precision(name="precision"),
                     keras.metrics.Recall(name="recall"), keras.metrics.AUC(name="pr_auc", curve="PR")])
mlp.summary()
historial = mlp.fit(X_aj, y_aj, validation_data=(X_val, y_val), epochs=20, batch_size=32, verbose=0)
historial_df = pd.DataFrame(historial.history)
mejor_epoca = int(historial_df["val_loss"].idxmin() + 1)
print("Mejor val_loss en época:", mejor_epoca)


Model: "MLP_AndinaLog_S9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ oculta_1 (Dense)                │ (None, 32)             │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ oculta_2 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ probabilidad_desviacion (Dense) │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,569 (6.13 KB)

 Trainable params: 1,569 (6.13 KB)

 Non-trainable params: 0 (0.00 B)

Mejor val_loss en época: 7


## 5. Comparación en validación

La MLP se compara con Dummy y con Random Forest de S7. El umbral 0,50 es solo una referencia preliminar.

In [7]:
# Evaluación preliminar solo en VALIDACIÓN; TEST permanece sellado.
prob_val = mlp.predict(X_val, verbose=0).ravel()
metricas_mlp = metricas(y_val, prob_val, .5)
s7 = pd.read_csv(RUTA_S7, encoding="utf-8-sig")
rf_s7 = s7.loc[s7["modelo"].eq("Random Forest Base")].iloc[0]
comparacion = pd.DataFrame([
    {"modelo": "Dummy mayoritaria", **metricas_dummy},
    {"modelo": "MLP inicial", **metricas_mlp},
    {"modelo": "Random Forest S7", "accuracy": rf_s7["accuracy"], "precision": rf_s7["precision"],
     "recall": rf_s7["recall"], "f1": rf_s7["f1"], "pr_auc": rf_s7["pr_auc"],
     "roc_auc": rf_s7["roc_auc"], "TN": rf_s7["TN"], "FP": rf_s7["FP"], "FN": rf_s7["FN"], "TP": rf_s7["TP"]},
])
print(comparacion.round(4).to_string(index=False))
print("TEST permanece sellado: se preparó la matriz, pero no se generaron predicciones.")


           modelo  accuracy  precision  recall     f1  pr_auc  roc_auc   TN  FP  FN  TP
Dummy mayoritaria    0.9648     0.0000  0.0000 0.0000  0.0352   0.5000 4136   0 151   0
      MLP inicial    0.9722     0.8478  0.2583 0.3959  0.3601   0.7480 4129   7 112  39
 Random Forest S7    0.9519     0.3333  0.3642 0.3481  0.3981   0.7635 4026 110  96  55
TEST permanece sellado: se preparó la matriz, pero no se generaron predicciones.


## 6. Evidencias

Se exportan curvas, matriz de confusión, historial y predicciones de validación. TEST no se consulta.

In [8]:
SALIDAS.mkdir(parents=True, exist_ok=True)
historial_df.insert(0, "epoca", np.arange(1, len(historial_df)+1))
historial_df.to_csv(SALIDAS / "historial_entrenamiento_mlp_s9.csv", index=False, encoding="utf-8-sig")
comparacion.to_csv(SALIDAS / "comparacion_validacion_s9.csv", index=False, encoding="utf-8-sig")

pred_val = (prob_val >= .5).astype(int)
predicciones = validacion[["fila_bronze", "viaje_id", "timestamp_bolivia", TARGET]].copy()
predicciones["probabilidad_mlp"] = prob_val
predicciones["prediccion_mlp_umbral_050"] = pred_val
predicciones.to_csv(SALIDAS / "predicciones_validacion_mlp_s9.csv", index=False, encoding="utf-8-sig")

epocas = historial_df["epoca"]
fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(epocas, historial_df["loss"], label="Ajuste")
axes[0].plot(epocas, historial_df["val_loss"], label="Validación")
axes[0].axvline(mejor_epoca, linestyle="--", color="gray", label="Mejor val_loss")
axes[0].set(title="Pérdida por época", xlabel="Época", ylabel="Binary crossentropy"); axes[0].legend()
axes[1].plot(epocas, historial_df["recall"], label="Ajuste")
axes[1].plot(epocas, historial_df["val_recall"], label="Validación")
axes[1].set(title="Recall por época", xlabel="Época", ylabel="Recall", ylim=(-.05,1.05)); axes[1].legend()
plt.tight_layout(); plt.savefig(SALIDAS / "curvas_aprendizaje_mlp_s9.png", dpi=150, bbox_inches="tight"); plt.close()

cm = confusion_matrix(y_val, pred_val, labels=[0,1])
fig, ax = plt.subplots(figsize=(5,4)); ax.imshow(cm, cmap="Blues")
for (i,j), v in np.ndenumerate(cm): ax.text(j,i,str(v),ha="center",va="center",color="white" if v>cm.max()/2 else "black")
ax.set(xticks=[0,1], yticks=[0,1], xlabel="Predicho", ylabel="Real", title="MLP inicial en validación · umbral 0,50")
plt.tight_layout(); plt.savefig(SALIDAS / "matriz_confusion_validacion_mlp_s9.png", dpi=150, bbox_inches="tight"); plt.close()
print("Salidas guardadas en", SALIDAS)


Salidas guardadas en c:\Users\remrodri\Github\practicasNotebookColab\proyecto-integrador\07_mlp\salidas_s9


## 7. Paso siguiente

S10 evaluará L2, Dropout y EarlyStopping usando validación. Solo después de elegir la configuración y el umbral se abrirá TEST para la MLP.